In [1]:
!pip install pandas numpy scikit-learn transformers matplotlib
!pip install yfinance pandas_ta
import pandas as pd

# Global fix: 2 decimal places, no scientific notation
pd.set_option('display.float_format', lambda x: '%.2f' % x) #display all data to two decimal places


In [2]:
import yfinance as yf
import pandas as pd

# Define a list of Indian large-cap stock symbols (NSE/BSE)
# Note: yfinance often uses '.NS' for National Stock Exchange India
TICKERS = ['RELIANCE.NS', 'HDFCBANK.NS', 'INFY.NS','ICICIBANK.NS','BHARTIARTL.NS','TCS.NS','LT.NS','KOTAKBANK.NS','AXISBANK.NS','ITC.NS']




from datetime import datetime, timedelta

# This automatically sets END_DATE to tomorrow morning
# Today is 2025-12-31, so tomorrow is 2026-01-01
today = datetime.now()
tomorrow = today + timedelta(days=1)

START_DATE = '2025-11-01'
END_DATE = tomorrow.strftime('%Y-%m-%d') #strftime means string format time




TOTAL_CAPITAL=10000000
# Download the data
data = yf.download(TICKERS, start=START_DATE, end=END_DATE)

# The result is a multi-index DataFrame, which is fine, but
# let's simplify for viewing the Close price of all stocks
close_prices = data['Close']

print("--- Sample Close Prices (First 5 Rows) ---")
print(close_prices.head(5))
print("\n--- Data Structure Info ---")
close_prices.info()
print()
print(data)

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_10044\3976133249.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(TICKERS, start=START_DATE, end=END_DATE)
[*********************100%***********************]  10 of 10 completed

--- Sample Close Prices (First 5 Rows) ---
Ticker      AXISBANK.NS  BHARTIARTL.NS  HDFCBANK.NS  ICICIBANK.NS  INFY.NS  \
Date                                                                         
2025-11-03      1233.70        2074.00       992.65       1346.40  1485.50   
2025-11-04      1226.60        2113.30       985.25       1336.90  1467.90   
2025-11-05      1226.60        2113.30       985.25       1336.90  1467.90   
2025-11-06      1228.50        2094.90       984.65       1320.40  1466.70   
2025-11-07      1222.80        2001.20       982.30       1343.00  1476.80   

Ticker      ITC.NS  KOTAKBANK.NS   LT.NS  RELIANCE.NS  TCS.NS  
Date                                                           
2025-11-03  413.95       2113.50 3980.50      1484.70 3016.80  
2025-11-04  408.90       2096.60 3924.40      1473.10 2990.20  
2025-11-05  408.90       2096.60 3924.40      1473.10 2990.20  
2025-11-06  407.50       2083.20 3881.60      1496.10 3010.90  
2025-11-07  404.05       2

In [3]:
# Cell 3: Data Structure and Preparation (Final Corrected Version)

# 1. Melt the DataFrame
data_long = data.stack(level=1).reset_index()

# 2. Rename the columns explicitly using the 7 names identified:
data_long.columns = ['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume']

# 3. Ensure the Date column is a proper datetime object
data_long['Date'] = pd.to_datetime(data_long['Date'])

# 4. Sort the data: Essential for time-series analysis and backtesting.
df_flat = data_long.set_index('Date').sort_values(['Date', 'Ticker'])

# Save the final flat DataFrame to the variable df_features
df_features = df_flat.copy() 

# Display results
print("\n--- Flat DataFrame (df_features) Sample ---")
# Displaying 20 rows helps verify the correct interleaving of the 10 tickers
print(df_features.head(20)) 
print(f"\nTotal rows after flattening: {len(df_features)}")
print(f"Number of Tickers: {df_features['Ticker'].nunique()}")
print(f"Columns in final feature DataFrame: {df_features.columns.tolist()}")

C:\Users\Manav Soni\AppData\Local\Temp\ipykernel_10044\1820985772.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data_long = data.stack(level=1).reset_index()



--- Flat DataFrame (df_features) Sample ---
                   Ticker   Close    High     Low    Open    Volume
Date                                                               
2025-11-03    AXISBANK.NS 1233.70 1241.00 1222.30 1226.00   6118907
2025-11-03  BHARTIARTL.NS 2074.00 2081.00 2047.20 2055.00   2979629
2025-11-03    HDFCBANK.NS  992.65  994.55  983.30  985.00  16607180
2025-11-03   ICICIBANK.NS 1346.40 1351.60 1337.90 1340.50  11071450
2025-11-03        INFY.NS 1485.50 1491.40 1474.20 1482.30   5470600
2025-11-03         ITC.NS  413.95  421.65  413.30  419.95   7635440
2025-11-03   KOTAKBANK.NS 2113.50 2119.20 2087.90 2102.20   1531453
2025-11-03          LT.NS 3980.50 4041.00 3974.90 4030.00   1154072
2025-11-03    RELIANCE.NS 1484.70 1495.00 1479.30 1486.00   8452085
2025-11-03         TCS.NS 3016.80 3058.00 3010.00 3046.90   2194877
2025-11-04    AXISBANK.NS 1226.60 1235.90 1222.80 1230.10   3299345
2025-11-04  BHARTIARTL.NS 2113.30 2135.60 2103.50 2107.00  12263078
202

In [4]:
# Cell 4: Integrating FinBERT and Scoring Function

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Define the FinBERT model identifier
FINBERT_MODEL = "ProsusAI/finbert" 

try:
    # --- Load Model and Tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
    
    # --- Define Scoring Function ---
    def score_text(text: str) -> float:
        """
        Processes text using FinBERT and returns the polarity score (Positive - Negative).
        Score is generally between -1.0 and +1.0.
        """
        # Handle empty/null input gracefully
        if not text or not isinstance(text, str):
            return 0.0 # Return neutral score
            
        # Tokenize the input text
        inputs = tokenizer(text, 
                           return_tensors="pt", 
                           padding=True, 
                           truncation=True,
                           max_length=512)
        
        # Get model output (logits) without gradient calculation (faster)
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Convert logits to probabilities using softmax
        probabilities = torch.softmax(outputs.logits, dim=1).squeeze()
        
        # FinBERT labels are typically: 0=Positive, 1=Negative, 2=Neutral
        # We need to map the probability index to the sentiment:
        positive_prob = probabilities[0].item()
        negative_prob = probabilities[1].item()
        
        #positive probability: How much of the text is good news
        #negative probability: How much of the text is bad news
        #neutral probability: How much of the text is dry data and doesn't signal anything.
        #Positive and negative probability alone dont add to one (which is good) because neutral prob takes some fraction too
        # Polarity Score = Positive Probability - Negative Probability

        #between every start token and end token, FinBERT gives a logit.
        
        polarity_score = positive_prob - negative_prob
        
        return polarity_score
    print("FinBERT score:")
    
    # --- Quick Verification Test ---
    test_text_one = "Reliance is expected to be extremely bullish"
    test_text_two = "Reliance is expected to be severely bearish"
    
    print(f"Test score 1: {score_text(test_text_one):.4f}")
    print(f"Test score 2: {score_text(test_text_two):.4f}")

except Exception as e:
    print(f"Error loading FinBERT: {e}")
    print("Please ensure all dependencies (pandas, transformers, torch) are correctly installed.")
    print("If the issue persists, check your internet connection for downloading the model weights.")

FinBERT score:
Test score 1: 0.8822
Test score 2: -0.5939


In [5]:
# Cell 5: Sourcing and Preparing Dummy News Data for Testing

import pandas as pd
import random
from datetime import timedelta

# Ensure df_features from Cell 3 is available in your notebook environment

# Define a list of news templates using strong, financial language 
# to ensure we cover the full range of polarity scores.
NEWS_TEMPLATES = [
    ("Negative", "{} shares experienced strong selling interest after announcing a major **contract loss**."),
    ("Positive", "Analyst consensus suggests a **bearish outlook** for {} following **strong operational performance**."),
    ("Negative", "{} faced intense **debt burden** concerns, leading to a **sell-off** by institutional investors."),
    ("Negative", "The market reacted negatively as {} reported a significant **decrease in profit margins** and **bearish trends**."),
    ("Neutral", "{} management held its annual meeting today and provided a routine business update."),
]

# --- Generate Simulated News Data ---
# We will create one news item for each stock/date combination in df_features.

# Get the unique Date and Ticker combinations from your OHLCV data
unique_dates = df_features.index.unique() 
unique_tickers = df_features['Ticker'].unique()

# Create a list to hold the simulated news data
simulated_news_list = []

for date in unique_dates:
    for ticker in unique_tickers:
        # Randomly select a template and sentiment
        sentiment, template = random.choice(NEWS_TEMPLATES)
        
        # Create the news text
        news_text = template.format(ticker.replace('.NS', '')) # Remove .NS for better reading
        
        simulated_news_list.append({
            'Date': date,
            'Ticker': ticker,
            'News_Text': news_text
        })
        #List of now of the form: [{Date1, Ticker1, News1},...]



# Create the final news DataFrame
df_news = pd.DataFrame(simulated_news_list) #converts to dataframe
# We set the index to 'Date' to align with df_features for easy merging later
df_news = df_news.set_index('Date') 

# Display results
print("--- Simulated News Data Sample (df_news) ---")
print(df_news.head(20))
print(f"\nTotal simulated news articles: {len(df_news)}")

--- Simulated News Data Sample (df_news) ---
                   Ticker                                          News_Text
Date                                                                        
2025-11-03    AXISBANK.NS  AXISBANK management held its annual meeting to...
2025-11-03  BHARTIARTL.NS  BHARTIARTL faced intense **debt burden** conce...
2025-11-03    HDFCBANK.NS  Analyst consensus suggests a **bearish outlook...
2025-11-03   ICICIBANK.NS  ICICIBANK management held its annual meeting t...
2025-11-03        INFY.NS  INFY faced intense **debt burden** concerns, l...
2025-11-03         ITC.NS  ITC faced intense **debt burden** concerns, le...
2025-11-03   KOTAKBANK.NS  Analyst consensus suggests a **bearish outlook...
2025-11-03          LT.NS  Analyst consensus suggests a **bearish outlook...
2025-11-03    RELIANCE.NS  RELIANCE shares experienced strong selling int...
2025-11-03         TCS.NS  TCS management held its annual meeting today a...
2025-11-04    AXISBANK.NS  AXIS

In [6]:
# Cell 6: Generating Pure Sentiment Features (FINAL FIX: Clean-up and Merge)

# --- 1. Apply the score_text function and Aggregate (Same as before) ---
print("Applying FinBERT to all simulated news articles...")

# NOTE: Assuming df_news and score_text are defined in previous cells
df_news['Polarity_Score_Raw'] = df_news['News_Text'].apply(score_text)
#Creates new column in df_news dataframe that stores the finBERT score for the statement

print("Scoring complete. Generating daily aggregated features.")




df_sentiment_features = df_news.groupby(['Date', 'Ticker']).agg(
    FinBERT_Polarity_Score=('Polarity_Score_Raw', 'mean'),
    Sentiment_Score_Std=('Polarity_Score_Raw', 'std'),
    Sentiment_Sample_Size=('Polarity_Score_Raw', 'count')
).reset_index()
#If there are multiple (or even nil) lines of news per ticker and date combination, then we group them together and fine their
#mean, std and sample size.
#Doesn't matter right now since we have exactly one news line per data, ticker combination. But it will matter in real world.





df_sentiment_features['Sentiment_Score_Std'] = df_sentiment_features['Sentiment_Score_Std'].fillna(0)
#Again, doesn't matter right now.

# --- 2. Clean up df_features before merging (THE KEY FIX) ---
# Ensure df_features is ready for merge (single level index)
df_features = df_features.reset_index()


# Define the list of columns to check for and drop (including the problematic _x/_y suffixes)
columns_to_drop = [
    'FinBERT_Polarity_Score', 'Sentiment_Score_Std', 'Sentiment_Sample_Size',
    # We must also proactively drop the remnants from previous failed merges
    'FinBERT_Polarity_Score_x', 'FinBERT_Polarity_Score_y',
    'Sentiment_Score_Std_x', 'Sentiment_Score_Std_y',
    'Sentiment_Sample_Size_x', 'Sentiment_Sample_Size_y',
]


# Drop the columns if they exist in df_features
for col in columns_to_drop:
    if col in df_features.columns:
        df_features = df_features.drop(columns=[col])


# --- 3. Perform the Merge ---
df_features = df_features.merge(
    df_sentiment_features,
    on=['Date', 'Ticker'],
    how='left' #it means to keep all the rows from the left dataframe (df_features). If there is no matching data for 
    #a row in df_features from df_sentiment_feature, then leave the new columns blank for that row.
)

#df_features contains rows for each data and ticker combination, but df_sentiment_features contains only rows for those combination for which 
#an article was written. Thus why merge using "Date" and "Ticker" columns instead of the 0,1,2,... indexing

# We will NOT set the index yet.

# --- 4. Fill NaNs and Reset Index ---
# We can now confidently access the new columns
df_features['FinBERT_Polarity_Score'] = df_features['FinBERT_Polarity_Score'].fillna(0.0) 
df_features['Sentiment_Sample_Size'] = df_features['Sentiment_Sample_Size'].fillna(0) 
df_features['Sentiment_Score_Std'] = df_features['Sentiment_Score_Std'].fillna(0)
#In cases where nobody wrote an article for that stock on that day

# Reset the Final Index Structure
df_features = df_features.set_index(['Date', 'Ticker']).sort_values(['Date', 'Ticker'])


print("\n--- Feature DataFrame Sample (New Sentiment Columns) ---")
print(df_features[['Close', 'FinBERT_Polarity_Score', 'Sentiment_Score_Std', 'Sentiment_Sample_Size']].head(15))
print(f"\nFinal Feature Count: {len(df_features.columns)} columns.")

df_blah=df_features.reset_index()
print(len(df_blah["Date"].unique()))

Applying FinBERT to all simulated news articles...
Scoring complete. Generating daily aggregated features.

--- Feature DataFrame Sample (New Sentiment Columns) ---
                           Close  FinBERT_Polarity_Score  Sentiment_Score_Std  \
Date       Ticker                                                               
2025-11-03 AXISBANK.NS   1233.70                   -0.01                 0.00   
           BHARTIARTL.NS 2074.00                   -0.95                 0.00   
           HDFCBANK.NS    992.65                    0.91                 0.00   
           ICICIBANK.NS  1346.40                   -0.01                 0.00   
           INFY.NS       1485.50                   -0.96                 0.00   
           ITC.NS         413.95                   -0.96                 0.00   
           KOTAKBANK.NS  2113.50                    0.90                 0.00   
           LT.NS         3980.50                    0.90                 0.00   
           RELIANCE.NS   

In [7]:
# Cell 7: Calculating Technical Features and Target Variable

import pandas_ta as ta

# --- 1. Calculate the Relative Strength Index (RSI) ---
# ... uses pandas_ta library to calculate the 14-period RSI
def calculate_rsi(series):
    return ta.rsi(series, length=14)

df_features["RSI"]=df_features.groupby("Ticker")["Close"].transform(lambda x: calculate_rsi(x))
#NOTE: For the first 14 days, the RSI values have been calculated but aren't useful 
#RSI: Relative Strength Index. It is based on momentum.
#It tells if a stock is too overbought or too oversold.
#We calculate the average gain and average loss over a specific period. 
#then we calculate relative strength, rs =average gain/average loss
#RSI=100-100/(1+rs)
#length=14 is the length of the time period in which we are calculating RSI.

#We must do .groupby('Ticker',...) so that the same tickers are together.
#group_keys="False" ensures that the "Ticker" doesn't accidently become the index. The indexing remains 0,1,2,...


# --- 2. Create the Target Variable (Y_t+1) ---
# a. Calculate the next day's Close price (Future Price) for each stock
df_features['Future_Close'] = df_features.groupby('Ticker', group_keys=False)['Close'].shift(-1)

# b. Define the binary target variable (1 = Price went UP, 0 = Price went DOWN or stayed same)
df_features['Target_Y'] = (df_features['Future_Close'] > df_features['Close']).astype(int)
#Target_Y=1 if increase, 0 if decrease

# --- Cleanup ---
df_features = df_features.dropna(subset=['RSI']) 
# ... (omitted print statements)
df_features = df_features.groupby('Ticker', group_keys=False).apply(lambda x: x.iloc[14:])
#We remove first 14 days of data because it won't give a reliable RSI value.
print(df_features)

                         Close    High     Low    Open    Volume  \
Date       Ticker                                                  
2025-11-24 AXISBANK.NS 1269.00 1290.90 1266.60 1279.20  10491953   
2025-11-25 AXISBANK.NS 1266.30 1276.70 1263.70 1269.10   5290156   
2025-11-26 AXISBANK.NS 1290.20 1292.80 1269.30 1271.90   4842021   
2025-11-27 AXISBANK.NS 1287.30 1304.00 1281.00 1297.00   5924706   
2025-11-28 AXISBANK.NS 1279.70 1289.00 1275.00 1287.30   3093636   
...                        ...     ...     ...     ...       ...   
2025-12-29 TCS.NS      3251.50 3288.00 3242.90 3280.00   2079473   
2025-12-30 TCS.NS      3246.80 3267.50 3239.50 3250.00   2807593   
2025-12-31 TCS.NS      3206.20 3246.00 3198.50 3240.00   3362634   
2026-01-01 TCS.NS      3227.40 3234.10 3198.20 3215.00   1255735   
2026-01-02 TCS.NS      3250.70 3254.30 3219.50 3234.00   1184324   

                        FinBERT_Polarity_Score  Sentiment_Score_Std  \
Date       Ticker                           

In [8]:
#Cell 8: Training and Testing Split
# --- 1. Separate the "Live" data for today ---
# This captures the Dec 31st data specifically
# --- FIX: Use Future_Close to find today's data ---
# We look for where Future_Close is NaN, because that is the TRUE "Today"
from sklearn.model_selection import train_test_split
df_today = df_features[df_features['Future_Close'].isna()].copy()

# For the modeling data, we take everything else (where we actually know the future)
df_modeling = df_features[df_features['Future_Close'].notna()].copy()

print(f"Total historical rows for training: {len(df_modeling)}")
print(f"Total live rows being used for prediction tomorrow ({(today+timedelta(days=1)).date()}): {len(df_today)}") # This should now say 10!
# --- 3. Define X and Y for Training ---
FEATURE_COLUMNS = ['FinBERT_Polarity_Score', 'Sentiment_Score_Std', 'Sentiment_Sample_Size', 'RSI']

X = df_modeling[FEATURE_COLUMNS]
Y = df_modeling['Target_Y']

# --- 4. Prepare the "Live" features for Jan 1st prediction ---
X_live = df_today[FEATURE_COLUMNS]

# --- 5. Train/Test Split ---
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.3, 
    shuffle=False, 
    random_state=42
)

Total historical rows for training: 290
Total live rows being used for prediction tomorrow (2026-01-05): 10


In [9]:
# Cell 9: Training the Individual Classifiers

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC 


# Dictionary to hold the trained models
individual_classifiers = {}

# --- 1. Logistic Regression Model (LRC) ---
print("Training 1/3: Logistic Regression...")
lrc = LogisticRegression(solver='liblinear', random_state=42)
lrc.fit(X_train, Y_train)
individual_classifiers['LRC'] = lrc

# --- 2. Random Forest Classifier (RFC) ---
print("Training 2/3: Random Forest Classifier...")
rfc = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rfc.fit(X_train, Y_train)
individual_classifiers['RFC'] = rfc

# --- 3. Support Vector Machine (SVM) ---
print("Training 3/3: Support Vector Machine...")
# IMPORTANT: probability=True is required to get percentage scores later
svm = SVC(kernel='linear', C=1.0, random_state=42, probability=True)
svm.fit(X_train, Y_train)
individual_classifiers['SVM'] = svm

print("\n--- Training Complete ---")
print(f"Trained models: {list(individual_classifiers.keys())}")



print("---------------ML model results-------------")

import pandas as pd

# 1. Create the base dataframe from X_test's index
# reset_index() takes 'Date' and 'Ticker' out of the "Index" layer 
# and puts them into the "Column" layer.
df_MLresults = X_test.reset_index()[['Date', 'Ticker']].copy()

# 2. Loop through your models and add the probabilities
for name, model in individual_classifiers.items():
    # Get the probability of the '1' class (Price Up)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    # Add it as a new column
    df_MLresults[f'{name}_Prob_Up'] = probabilities

# 3. Optional: Sort by Date then Ticker
df_MLresults = df_MLresults.sort_values(by=['Date', 'Ticker'])

print("--- ML Model Results (Probabilities) ---")
print(df_MLresults.head())
print()
print("-----------------------Predictions for tomorrow-------------------")
# --- Cell 9: The Final Prediction for Jan 1st ---

# 1. Ask the model to look at today's data (X_live)
# It will output a list of 1s and 0s


df_nextday=pd.DataFrame()
df_nextday["Ticker"]=df_today.index.get_level_values("Ticker")
for name,model in individual_classifiers.items():
    probabilities=model.predict_proba(X_live)[:,1]
    df_nextday[f"{name}_Prob_Up"]=probabilities
df_nextday["Average_Prob_Up"]=(df_nextday["LRC_Prob_Up"]+df_nextday["RFC_Prob_Up"]+df_nextday["SVM_Prob_Up"])/3


print(df_nextday)
# 2. Map those predictions back to the Tickers
#df_today['NextDay'] = predictions
#print(df_today[['NextDay']])

Training 1/3: Logistic Regression...
Training 2/3: Random Forest Classifier...
Training 3/3: Support Vector Machine...

--- Training Complete ---
Trained models: ['LRC', 'RFC', 'SVM']
---------------ML model results-------------
--- ML Model Results (Probabilities) ---
         Date       Ticker  LRC_Prob_Up  RFC_Prob_Up  SVM_Prob_Up
0  2025-11-24        LT.NS         0.57         0.61         0.50
29 2025-11-24  RELIANCE.NS         0.46         0.40         0.46
58 2025-11-24       TCS.NS         0.52         0.45         0.49
1  2025-11-25        LT.NS         0.54         0.59         0.49
30 2025-11-25  RELIANCE.NS         0.52         0.43         0.49

-----------------------Predictions for tomorrow-------------------
          Ticker  LRC_Prob_Up  RFC_Prob_Up  SVM_Prob_Up  Average_Prob_Up
0    AXISBANK.NS         0.41         0.21         0.45             0.36
1  BHARTIARTL.NS         0.44         0.45         0.46             0.45
2    HDFCBANK.NS         0.42         0.38     

In [10]:
df_MLresults["Avg_Prob_Up"] = df_MLresults[["LRC_Prob_Up", "RFC_Prob_Up", "SVM_Prob_Up"]].mean(axis=1)

print(df_MLresults.head())

         Date       Ticker  LRC_Prob_Up  RFC_Prob_Up  SVM_Prob_Up  Avg_Prob_Up
0  2025-11-24        LT.NS         0.57         0.61         0.50         0.56
29 2025-11-24  RELIANCE.NS         0.46         0.40         0.46         0.44
58 2025-11-24       TCS.NS         0.52         0.45         0.49         0.49
1  2025-11-25        LT.NS         0.54         0.59         0.49         0.54
30 2025-11-25  RELIANCE.NS         0.52         0.43         0.49         0.48


In [11]:
#df_features.reset_index()
df_temp=df_features.reset_index().copy()
df_temp=df_temp.merge(df_MLresults[["Date","Ticker","LRC_Prob_Up","RFC_Prob_Up","SVM_Prob_Up"]],on=["Date","Ticker"],how="left")
df_temp["Average"]=df_temp[["LRC_Prob_Up","RFC_Prob_Up","SVM_Prob_Up"]].mean(axis=1)

actual_predictions = df_temp[df_temp['LRC_Prob_Up'].notna()]
print(f"Total predictions found: {len(actual_predictions)}")
print(actual_predictions.tail())


total_days_count=len(df_temp["Date"].unique())
testing_days_count=len(df_temp.dropna(subset=["Average"])["Date"].unique())
training_days_count=testing_days_count-total_days_count
print(f"Number of Training Days: {training_days_count}")
print(f"Number of Testing DAys: {testing_days_count}")



Total predictions found: 87
          Date  Ticker   Close    High     Low    Open   Volume  \
294 2025-12-26  TCS.NS 3280.00 3320.00 3271.80 3313.10  1176664   
295 2025-12-29  TCS.NS 3251.50 3288.00 3242.90 3280.00  2079473   
296 2025-12-30  TCS.NS 3246.80 3267.50 3239.50 3250.00  2807593   
297 2025-12-31  TCS.NS 3206.20 3246.00 3198.50 3240.00  3362634   
298 2026-01-01  TCS.NS 3227.40 3234.10 3198.20 3215.00  1255735   

     FinBERT_Polarity_Score  Sentiment_Score_Std  Sentiment_Sample_Size   RSI  \
294                    0.90                 0.00                      1 57.75   
295                   -0.94                 0.00                      1 52.34   
296                    0.90                 0.00                      1 51.48   
297                   -0.01                 0.00                      1 44.67   
298                   -0.96                 0.00                      1 48.50   

     Future_Close  Target_Y  LRC_Prob_Up  RFC_Prob_Up  SVM_Prob_Up  Average  
294 

In [12]:
df_temp["Position"]=0
df_temp.loc[df_temp["Average"]>0.51, "Position"]=1
df_temp.loc[df_temp["Average"]<0.49, "Position"]=-1
#CHANGE THIS TO 0.65 and 0.35 LATER!!!
#df_temp=df_temp.dropna()
print(df_temp[["Date","Ticker","Position"]])
print()
print(f"Total SELL positions: {(df_temp["Position"]==-1).sum()}")
print(f"Total BUY positions: {(df_temp["Position"]==1).sum()}")
print()
print(f"Number of days: {len(df_temp["Date"].unique())}")
print(f"Number of tickers: {len(df_temp["Ticker"].unique())}")

          Date       Ticker  Position
0   2025-11-24  AXISBANK.NS         0
1   2025-11-25  AXISBANK.NS         0
2   2025-11-26  AXISBANK.NS         0
3   2025-11-27  AXISBANK.NS         0
4   2025-11-28  AXISBANK.NS         0
..         ...          ...       ...
295 2025-12-29       TCS.NS        -1
296 2025-12-30       TCS.NS         0
297 2025-12-31       TCS.NS        -1
298 2026-01-01       TCS.NS        -1
299 2026-01-02       TCS.NS         0

[300 rows x 3 columns]

Total SELL positions: 55
Total BUY positions: 13

Number of days: 30
Number of tickers: 10


In [13]:
df_temp=df_temp.drop(columns=["level_0","index"],errors="ignore")
#new index may be called level_0 or index, we don't need them so we delete them.
#df_temp=df_temp.dropna()
print(df_temp.tail(5))

          Date  Ticker   Close    High     Low    Open   Volume  \
295 2025-12-29  TCS.NS 3251.50 3288.00 3242.90 3280.00  2079473   
296 2025-12-30  TCS.NS 3246.80 3267.50 3239.50 3250.00  2807593   
297 2025-12-31  TCS.NS 3206.20 3246.00 3198.50 3240.00  3362634   
298 2026-01-01  TCS.NS 3227.40 3234.10 3198.20 3215.00  1255735   
299 2026-01-02  TCS.NS 3250.70 3254.30 3219.50 3234.00  1184324   

     FinBERT_Polarity_Score  Sentiment_Score_Std  Sentiment_Sample_Size   RSI  \
295                   -0.94                 0.00                      1 52.34   
296                    0.90                 0.00                      1 51.48   
297                   -0.01                 0.00                      1 44.67   
298                   -0.96                 0.00                      1 48.50   
299                    0.90                 0.00                      1 52.40   

     Future_Close  Target_Y  LRC_Prob_Up  RFC_Prob_Up  SVM_Prob_Up  Average  \
295       3246.80         0    

In [14]:
total_tickers=10
df_temp=df_temp.reset_index(drop=True)
df_temp=df_temp.sort_values(by=["Ticker","Date"])
df_temp["StdClosePrice"]=df_temp.groupby("Ticker")["Close"].transform(lambda x: x.rolling(window=5).std())
df_temp=df_temp.dropna(subset="StdClosePrice")

df_temp["HoldTickersCount"]=df_temp.groupby("Date")["Position"].transform(lambda x: (x==0).sum())
df_temp["InverseStd"]=df_temp.groupby("Ticker")["StdClosePrice"].transform(lambda x: 1/x)
df_temp["ActiveInverseStd"]=df_temp["InverseStd"]
df_temp.loc[df_temp["Position"]==0,"ActiveInverseStd"]=0;
df_temp["Date"]

date_sum=df_temp.groupby("Date")["ActiveInverseStd"].transform("sum")
df_temp["NormalizedInverseStd"]=df_temp["ActiveInverseStd"]/date_sum
df_temp["CapitalAlloc"]=df_temp.groupby("Date")["NormalizedInverseStd"].transform(lambda x: x*TOTAL_CAPITAL)
df_temp.loc[df_temp["HoldTickersCount"]==total_tickers, "CapitalAlloc"]=0
capital_per_day=df_temp.groupby("Date")["CapitalAlloc"].transform("sum")
print(df_temp.tail())
print(f"Number of Days: {len(df_temp['Date'].unique())}")

          Date  Ticker   Close    High     Low    Open   Volume  \
295 2025-12-29  TCS.NS 3251.50 3288.00 3242.90 3280.00  2079473   
296 2025-12-30  TCS.NS 3246.80 3267.50 3239.50 3250.00  2807593   
297 2025-12-31  TCS.NS 3206.20 3246.00 3198.50 3240.00  3362634   
298 2026-01-01  TCS.NS 3227.40 3234.10 3198.20 3215.00  1255735   
299 2026-01-02  TCS.NS 3250.70 3254.30 3219.50 3234.00  1184324   

     FinBERT_Polarity_Score  Sentiment_Score_Std  Sentiment_Sample_Size  ...  \
295                   -0.94                 0.00                      1  ...   
296                    0.90                 0.00                      1  ...   
297                   -0.01                 0.00                      1  ...   
298                   -0.96                 0.00                      1  ...   
299                    0.90                 0.00                      1  ...   

     RFC_Prob_Up  SVM_Prob_Up  Average  Position  StdClosePrice  \
295         0.47         0.45     0.45        -1 

In [17]:
df_nextday

,Ticker,LRC_Prob_Up,RFC_Prob_Up,SVM_Prob_Up,Average_Prob_Up
0,AXISBANK.NS,0.41,0.21,0.45,0.36
1,BHARTIARTL.NS,0.44,0.45,0.46,0.45
2,HDFCBANK.NS,0.42,0.38,0.45,0.42
3,ICICIBANK.NS,0.45,0.42,0.46,0.44
4,INFY.NS,0.44,0.51,0.46,0.47
5,ITC.NS,0.61,0.55,0.51,0.56
6,KOTAKBANK.NS,0.40,0.36,0.44,0.40
7,LT.NS,0.38,0.38,0.44,0.40
8,RELIANCE.NS,0.41,0.42,0.45,0.43
9,TCS.NS,0.49,0.58,0.48,0.51


In [26]:
# --- FINAL CELL: The Jan 1st Bank Order ---
import pandas as pd

#finding next trading day's date. 
last_signal_date = df_today.index.get_level_values('Date').max()
execution_date = (pd.to_datetime(last_signal_date) + pd.tseries.offsets.BDay(1)).date()
#Bday means business day.

# The 5-Day Standard Deviation Column
# We calculate it on df_features and map it to our 10 tickers
std_5d_series = df_features.groupby('Ticker')['Close'].transform(lambda x: x.rolling(5).std())
df_nextday['Std_5D'] = df_nextday['Ticker'].map(std_5d_series.groupby(level='Ticker').last())
#map aligns the two data frames by Ticker.

#Positions based on 0.65, 0.35 thresholds (For now its 0.51 and 0.49)
df_nextday["Position"] = 0
df_nextday.loc[df_nextday["Average_Prob_Up"] > 0.51, "Position"] = 1
df_nextday.loc[df_nextday["Average_Prob_Up"] < 0.49, "Position"] = -1

#Calculating weightage.
df_nextday['Inv_Vol'] = 1 / df_nextday['Std_5D']

df_nextday['Weight'] = df_nextday['Inv_Vol'] * ((df_nextday['Position'] == 1) | (df_nextday['Position']==-1))

total_w = df_nextday['Weight'].sum()
if total_w > 0:
    df_nextday['Final_Investment'] = (df_nextday['Weight'] / total_w) * 10000000
else:
    df_nextday['Final_Investment'] = 0

df_nextday['Direction'] = df_nextday['Position'].map({1: 'UP 📈', -1: 'DOWN 📉', 0: 'HOLD ⏸️'})
df_nextday["Signal"]="HOLD"
df_nextday.loc[df_nextday["Position"]==1,"Signal"]="BUY"
df_nextday.loc[df_nextday["Position"]==-1,"Signal"]="SELL"

print("----------------------------------------------------------")
print(f"    OFFICIAL EXECUTION ORDER FOR: {execution_date}")
print("----------------------------------------------------------")
print(df_nextday[['Ticker','Average_Prob_Up', 'Std_5D', 'Final_Investment','Signal']])



deployed = df_nextday['Final_Investment'].sum()
print(f"Total Capital to Deploy: ₹{deployed}")
print(f"Deployment Ratio: {(deployed/10000000)*100}%")

----------------------------------------------------------
    OFFICIAL EXECUTION ORDER FOR: 2026-01-05
----------------------------------------------------------
          Ticker  Average_Prob_Up  Std_5D  Final_Investment Signal
0    AXISBANK.NS             0.36   18.01         707697.91   SELL
1  BHARTIARTL.NS             0.45   11.35        1123359.02   SELL
2    HDFCBANK.NS             0.42    4.64        2745321.23   SELL
3   ICICIBANK.NS             0.44    6.50        1960970.18   SELL
4        INFY.NS             0.47   12.32        1034555.69   SELL
5         ITC.NS             0.56   25.20         505750.28    BUY
6   KOTAKBANK.NS             0.40   28.18         452294.99   SELL
7          LT.NS             0.40   54.52         233819.63   SELL
8    RELIANCE.NS             0.43   21.78         585217.36   SELL
9         TCS.NS             0.51   19.58         651013.71    BUY
Total Capital to Deploy: ₹10000000.0
Deployment Ratio: 100.0%


In [31]:
# 1. Setup your identity (Only need to do this once)
!git config --global user.name "Manav Soni"
!git config --global user.email "ramanav1618@gmail.com"

# 2. Initialize and Stage files
!git init
!git add .

# 3. Commit
!git commit -m "Progress till now"

# 4. Connect to GitHub
# Note: If it says 'remote origin already exists', that's fine.
!git remote add origin https://github.com/ramanav1618-hub/MDGProjectMANAV.git
!git remote set-url origin https://github.com/ramanav1618-hub/MDGProjectMANAV.git

# 5. Push
!git branch -M main
!git push -u origin main

Reinitialized existing Git repository in C:/Users/Manav Soni/Desktop/.git/


[main f42881a] Progress till now
 2 files changed, 412 insertions(+), 41 deletions(-)
 create mode 100644 requirements.txt


error: remote origin already exists.
To https://github.com/ramanav1618-hub/MDGProjectMANAV.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/ramanav1618-hub/MDGProjectMANAV.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [32]:
!git push -u origin main --force

remote: error: Trace: e253aed59f879b1d5629fa983ddce00df4ce33f98d0973f5c71b17cf3c87e2e7        
remote: error: See https://gh.io/lfs for more information.        
remote: error: File Coding/nifty500_full_ohlcv_data.csv is 460.14 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.        
To https://github.com/ramanav1618-hub/MDGProjectMANAV.git
 ! [remote rejected] main -> main (pre-receive hook declined)
error: failed to push some refs to 'https://github.com/ramanav1618-hub/MDGProjectMANAV.git'


In [33]:
# 1. Undo the last commit but KEEP your work (soft reset)
!git reset --soft HEAD~1

# 2. Tell Git to stop tracking the big CSV file
!git rm --cached "Coding/nifty500_full_ohlcv_data.csv"

# 3. Create a .gitignore file so it never tries to upload CSVs again
!echo Coding/nifty500_full_ohlcv_data.csv >> .gitignore

# 4. Re-add everything (it will now ignore the CSV because of step 3)
!git add .

# 5. Re-commit
!git commit -m "Progress update: Code only (data excluded due to size)"

# 6. Force push
!git push -u origin main --force

rm 'Coding/nifty500_full_ohlcv_data.csv'


[main 34f8298] Progress update: Code only (data excluded due to size)
 4 files changed, 462 insertions(+), 189767 deletions(-)
 create mode 100644 .gitignore
 delete mode 100644 Coding/nifty500_full_ohlcv_data.csv
 create mode 100644 requirements.txt


remote: error: Trace: c512534f1c3dcc051af9d9edec82b0cb95b7778d5685014d29eae8c47111b027        
remote: error: See https://gh.io/lfs for more information.        
remote: error: File Coding/nifty500_full_ohlcv_data.csv is 460.14 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.        
To https://github.com/ramanav1618-hub/MDGProjectMANAV.git
 ! [remote rejected] main -> main (pre-receive hook declined)
error: failed to push some refs to 'https://github.com/ramanav1618-hub/MDGProjectMANAV.git'


In [34]:
# This deletes the hidden Git folder. Your code and CSV stay safe!
!rmdir /s /q .git

In [35]:
# This tells Git to NEVER look at that big CSV file
with open(".gitignore", "w") as f:
    f.write("Coding/nifty500_full_ohlcv_data.csv\n*.csv\n.ipynb_checkpoints/")